# Stage 6 · Final frozen test evaluation

This notebook reads the archived final test pass for YOLO26s. The checkpoint, thresholds, and size cutoffs were frozen before the test images were evaluated.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from roadguard.utils.paths import project_root

root = project_root()
selected = os.environ.get("ROADGUARD_STAGE6_RUN")
if selected:
    run = Path(selected)
    result_path = run / "stage6_result.json"
else:
    result_path = root / "artifacts/experiments/stage6_result.json"
    run = root / json.loads(result_path.read_text(encoding="utf-8"))["run"]
result = json.loads(result_path.read_text(encoding="utf-8"))
metadata = json.loads((run / "metadata.json").read_text(encoding="utf-8"))
protocol = json.loads((run / "frozen_protocol.json").read_text(encoding="utf-8"))
print(run.name)

## Scope and fixed evaluation protocol

Ultralytics computes AP across confidence values at the frozen evaluation floor. Box diagnostics use the existing 0.25 operating threshold and 0.50 matching IoU.

In [ ]:
print(f"Test images: {result['test_images']:,}")
print(f"Checkpoint epoch: {metadata['checkpoint_epoch']}")
print("Test evaluation passes:", result["integrity"]["test_evaluation_passes"])
print("Training occurred:", result["integrity"]["training_occurred"])
settings = metadata["identity"].copy()
environment = settings.pop("environment")
display(pd.Series(settings, name="Frozen test settings").to_frame())
display(pd.Series(environment, name="Recorded runtime").to_frame())

## Generalization from validation to test

Global differences use the published Stage 4 reference values, including 0.3234 for mAP50–95. The per-class comparison uses the full-precision Stage 4 archive.

In [ ]:
comparison = result["comparison"]
display(pd.DataFrame(comparison["global_metrics"]).drop(columns="absolute_magnitude").round(4))
display(pd.DataFrame(comparison["per_class_ap50_95"]).round(4))
print(comparison["generalization_assessment"])
print(f"Test mAP50–95 gap: {comparison['generalization_gap_percentage_points']:+.2f} pp")

## Test performance by class

The confusion matrix reports counts and shares of ground-truth rows. The background column includes defects without a sufficiently overlapping paired detection.

In [ ]:
classes = pd.DataFrame(result["per_class"])
fields = [
    "class",
    "evaluated_annotations",
    "precision",
    "recall",
    "ap50",
    "ap50_95",
    "false_positives",
    "false_negatives",
]
display(classes[fields].round(4))
display(Image(filename=str(run / "figures/confusion_matrix.png"), width=820))
display(Image(filename=str(run / "BoxPR_curve.png"), width=820))

## Performance by size

The lower and upper area cutoffs come from Stage 5 validation. A larger miss rate for small defects shows that the weakness persists on the held-out split.

In [ ]:
sizes = pd.DataFrame(result["size_analysis"])
display(sizes.loc[sizes["class"] == "All"].drop(columns="class").round(4))
size_gaps = pd.DataFrame(
    {
        "Validation miss rate": comparison["validation_miss_rate_by_size"],
        "Test miss rate": comparison["test_miss_rate_by_size"],
    }
)
display(size_gaps.round(4))
display(Image(filename=str(run / "figures/object_size.png"), width=780))

## Examples from the final test pass

The left panel shows annotations. The right panel shows predictions at the fixed operating threshold.

In [ ]:
gallery = json.loads((run / "gallery_index.json").read_text(encoding="utf-8"))
for category in ("correct_detections", "false_positives", "small_missed_defects"):
    print(category.replace("_", " ").title())
    display(Image(filename=str(run / gallery[category][0]["figure"]), width=960))

## Detector decision

Accepted with limitations for research. Small-defect recall remains a limitation. Test-set inference characterizes the detector under a fixed offline protocol. Camera, field conditions, telemetry, and end-to-end system latency require separate vehicle validation.

In [ ]:
print("Strongest by test AP50–95:", result["strongest_class"])
print("Weakest by test AP50–95:", result["weakest_class"])
print("Test small-defect miss rate:", f"{comparison['test_miss_rate_by_size']['small']:.1%}")
print("Checkpoint unchanged:", result["integrity"]["checkpoint_unchanged"])
print("Verified partitions:", ", ".join(result["integrity"]["verified_checksum_splits"]))